# Pulling Options Data from TWS — A Step-by-Step Walkthrough

This notebook explains **how** and **why** each step works when fetching futures options data from Interactive Brokers TWS using `ib_insync`.

Run the cells in order. TWS must be open and logged in before you start.

---

## The Big Picture

Fetching options data from TWS is a two-phase process:

```
Phase 1 — DISCOVER (no market data, fast)
  Connect → Qualify underlying → reqSecDefOptParams
  → Returns: list of valid (expiry, strike) pairs

Phase 2 — FETCH (market data, slower)
  Build FuturesOption contracts → qualifyContracts
  → reqTickers → bid / ask / IV / greeks
```

Phase 1 is cheap (milliseconds, no data subscription needed).  
Phase 2 costs market data subscription credits and takes a few seconds per batch.

## Prerequisites

Before running this notebook:

1. **TWS is open** and you are logged in (paper or live account)
2. **API is enabled** in TWS:  
   `Edit → Global Configuration → API → Settings`  
   ✅ Enable ActiveX and Socket Clients  
   ❌ Read-Only API (uncheck — our code enforces read-only at the protocol level)
3. `ib_insync` and `pandas` are installed (`pip install ib_insync pandas plotly`)

---
## Step 0 — Imports

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ib_insync import IB, Future, FuturesOption, util

# Suppress ib_insync's own verbose logger — we'll print what we need
util.logToConsole(False)

print('Imports OK')

---
## Step 1 — Connect to TWS

`IB()` creates a client object. `.connect()` opens a socket to TWS.

Key parameters:
| Parameter | Meaning |
|---|---|
| `host` | Where TWS is running. `127.0.0.1` = this machine |
| `port` | `7497` TWS paper \| `7496` TWS live \| `4002` Gateway paper \| `4001` Gateway live |
| `clientId` | Any integer. Each connection needs a unique ID. |
| `readonly` | **True** = TWS will reject any order attempt at the protocol level |

You can have multiple simultaneous connections (different clientIds) — useful for separating a data feed from a risk monitor.

In [ ]:
PORT = 7497   # change to 7496 for live

ib = IB()
ib.connect('127.0.0.1', PORT, clientId=99, readonly=True)

print(f'Connected: {ib.isConnected()}')
print(f'Server version: {ib.client.serverVersion()}')

---
## Step 2 — Define the Underlying Futures Contract

We start with the **underlying futures contract**, not the options. Options are defined relative to a specific futures contract.

`Future(symbol, exchange, currency)` creates an unqualified contract — it's just a description.  
IB doesn't assign a `conId` yet.

Common symbols:
| Symbol | Name | Exchange |
|---|---|---|
| ZC | Corn | CBOT |
| ZS | Soybeans | CBOT |
| ES | E-mini S&P 500 | CME |
| CL | Crude Oil | NYMEX |
| GC | Gold | COMEX |

In [ ]:
# Define the underlying — ZC = CBOT Corn futures
# Leaving lastTradeDateOrContractMonth empty tells IB to give us the front month
raw_contract = Future(
    symbol='ZC',
    exchange='CBOT',
    currency='USD'
)

print('Before qualification:')
print(f'  conId      : {raw_contract.conId}')   # 0 — not yet assigned
print(f'  localSymbol: "{raw_contract.localSymbol}"')  # empty

---
## Step 3 — Qualify the Contract

`qualifyContracts()` sends the description to TWS, which looks up the exact contract in its database and fills in missing fields:
- `conId` — IB's unique internal contract ID (the most reliable identifier)
- `localSymbol` — the exchange-specific ticker (e.g. `ZCN6` for July 2026 corn)
- `lastTradeDateOrContractMonth` — the actual expiry date
- `multiplier` — contract size

If you pass an ambiguous contract (e.g. a stock with the same symbol on two exchanges), IB returns multiple matches — that's why the return value is always a list.

In [ ]:
qualified = ib.qualifyContracts(raw_contract)
underlying = qualified[0]   # front month

print('After qualification:')
print(f'  conId                        : {underlying.conId}')
print(f'  localSymbol                  : {underlying.localSymbol}')
print(f'  lastTradeDateOrContractMonth : {underlying.lastTradeDateOrContractMonth}')
print(f'  multiplier                   : {underlying.multiplier}')
print(f'  exchange                     : {underlying.exchange}')

---
## Step 4 — Get the Current Futures Price

We need the current price to know which strikes are near-the-money.

`reqTickers()` requests a market data snapshot for one or more contracts.  
It waits until data arrives (or times out) and returns a list of `Ticker` objects.

The `Ticker` has many fields — the most useful for a snapshot:
- `last` — last traded price
- `bid` / `ask` — current best bid/ask
- `close` — previous session close
- `volume` — day volume

In [ ]:
[und_ticker] = ib.reqTickers(underlying)

print(f'Last  : {und_ticker.last}')
print(f'Bid   : {und_ticker.bid}')
print(f'Ask   : {und_ticker.ask}')
print(f'Close : {und_ticker.close}')
print(f'Volume: {und_ticker.volume}')

# Use last price as ATM reference; fall back to close if no last
atm_price = und_ticker.last or und_ticker.close or 460.0
print(f'\nUsing ATM reference price: {atm_price}')
print('(ZC prices are in US cents per bushel — divide by 100 for USD)')

---
## Step 5 — Discover the Options Structure (`reqSecDefOptParams`)

This is Phase 1 — the cheap discovery step.

`reqSecDefOptParams()` asks TWS: *"what options exist on this underlying?"*  
It returns the **schema** of the options market — which expiry dates and which strikes IB lists — but **no prices**.

Parameters:
| Parameter | Value for corn |
|---|---|
| `underlyingSymbol` | `'ZC'` |
| `futFopExchange` | `''` (empty = return all exchanges) |
| `underlyingSecType` | `'FUT'` (futures options, not equity options) |
| `underlyingConId` | the conId we got from `qualifyContracts` |

The result is a list of `OptionChain` objects, one per exchange listing. Each has:
- `.expirations` — set of date strings (YYYYMMDD)
- `.strikes` — set of strike prices
- `.exchange` — which exchange this listing is on

In [ ]:
chains = ib.reqSecDefOptParams(
    underlyingSymbol='ZC',
    futFopExchange='',          # empty → all exchanges
    underlyingSecType='FUT',
    underlyingConId=underlying.conId,
)

print(f'Option chain listings returned: {len(chains)}')
for c in chains:
    print(f'  Exchange: {c.exchange:8s}  '
          f'Expiries: {len(c.expirations):3d}  '
          f'Strikes: {len(c.strikes):4d}')

---
## Step 6 — Flatten to a Tidy DataFrame

We filter to CBOT only (avoids duplicates from other exchanges that might list the same options) and expand the (expiry, strike) pairs into a flat table.

In [ ]:
# Keep CBOT listings only
cbot_chains = [c for c in chains if c.exchange == 'CBOT']
if not cbot_chains:
    cbot_chains = chains   # fallback if CBOT not listed separately

rows = [
    {'expiry': exp, 'strike': float(s)}
    for chain in cbot_chains
    for exp in chain.expirations
    for s in chain.strikes
]

params_df = pd.DataFrame(rows).drop_duplicates().sort_values(['expiry', 'strike'])

print(f'Total (expiry, strike) pairs: {len(params_df):,}')
print(f'Unique expiries             : {params_df["expiry"].nunique()}')
print(f'Unique strikes              : {params_df["strike"].nunique()}')
print()
print('First 5 expiries:', sorted(params_df['expiry'].unique())[:5])
print('Strike range:', params_df['strike'].min(), '→', params_df['strike'].max())

---
## Step 7 — Choose an Expiry and Filter Strikes Near ATM

Corn has **hundreds** of strikes per expiry. Fetching all of them would:
- Take a long time (IB paces market data requests)
- Use unnecessary market data bandwidth
- Return mostly useless deep OTM quotes with zero volume

We select the nearest N strikes centred on the ATM price.

In [ ]:
# Pick the nearest expiry
all_expiries = sorted(params_df['expiry'].unique())
chosen_expiry = all_expiries[0]
print(f'All available expiries (first 8): {all_expiries[:8]}')
print(f'Using: {chosen_expiry}')

# Get all strikes for this expiry
all_strikes = sorted(
    params_df[params_df['expiry'] == chosen_expiry]['strike'].tolist()
)
print(f'\nAll strikes for {chosen_expiry}: {len(all_strikes)} total')
print(f'Range: {all_strikes[0]} → {all_strikes[-1]}')

# Select 20 strikes centred around ATM
N = 20
atm_idx = min(range(len(all_strikes)), key=lambda i: abs(all_strikes[i] - atm_price))
lo = max(0, atm_idx - N // 2)
hi = min(len(all_strikes), atm_idx + N // 2)
selected_strikes = all_strikes[lo:hi]

print(f'\nATM price: {atm_price}')
print(f'Closest strike to ATM: {all_strikes[atm_idx]}')
print(f'Selected {len(selected_strikes)} strikes: {selected_strikes[0]} → {selected_strikes[-1]}')

---
## Step 8 — Build `FuturesOption` Contracts

Now we build a contract object for each individual option — one per (strike, right) combination.  
**Right** = `'C'` for call, `'P'` for put.

Key fields for `FuturesOption`:
| Field | Value | Why |
|---|---|---|
| `symbol` | `'ZC'` | Root futures symbol |
| `lastTradeDateOrContractMonth` | `'20260527'` | Must match exactly what `reqSecDefOptParams` returned |
| `strike` | e.g. `460.0` | The strike price |
| `right` | `'C'` or `'P'` | Call or put |
| `exchange` | `'CBOT'` | Where the option is listed |
| `currency` | `'USD'` | |
| `multiplier` | `'50'` | Corn: 5000 bu contract, quoted in cents → $50 per cent move |

In [ ]:
# Build one contract object per (strike, right)
contracts = [
    FuturesOption(
        symbol='ZC',
        lastTradeDateOrContractMonth=chosen_expiry,
        strike=strike,
        right=right,
        exchange='CBOT',
        currency='USD',
        multiplier='50',
    )
    for strike in selected_strikes
    for right in ('C', 'P')
]

print(f'Built {len(contracts)} unqualified option contracts')
print(f'  ({len(selected_strikes)} strikes × 2 rights = {len(contracts)})')
print()
# Peek at the first one
sample = contracts[len(contracts)//2]
print('Sample contract (unqualified):')
print(f'  symbol : {sample.symbol}')
print(f'  expiry : {sample.lastTradeDateOrContractMonth}')
print(f'  strike : {sample.strike}')
print(f'  right  : {sample.right}')
print(f'  conId  : {sample.conId}')   # still 0

---
## Step 9 — Qualify the Option Contracts

`qualifyContracts()` sends all our option descriptions to TWS for validation.  
TWS fills in the `conId` for each valid contract and **silently drops** any that don't exist.

Why would a contract not exist?
- The strike was in `reqSecDefOptParams` but IB no longer lists it (old/expired)
- The strike/expiry combination was from a different exchange listing
- The option simply hasn't been assigned a `conId` yet (very new listing)

Always check `len(qualified)` vs `len(contracts)` — the difference tells you how many were invalid.

In [ ]:
qualified = ib.qualifyContracts(*contracts)

print(f'Submitted  : {len(contracts)} contracts')
print(f'Qualified  : {len(qualified)} contracts')
print(f'Dropped    : {len(contracts) - len(qualified)} (invalid/unlisted)')

if qualified:
    sample_q = qualified[len(qualified)//2]
    print()
    print('Sample qualified contract:')
    print(f'  conId      : {sample_q.conId}')   # now filled in
    print(f'  localSymbol: {sample_q.localSymbol}')
    print(f'  strike     : {sample_q.strike}')
    print(f'  right      : {sample_q.right}')

---
## Step 10 — Request Market Data Snapshots

Now the expensive part: `reqTickers()` subscribes to market data for each contract,  
waits for a snapshot, and returns.

**IB paces market data requests** — sending too many at once causes errors.  
We process in batches of 50 with a short sleep between batches.

What comes back in each `Ticker`:
- `bid`, `ask`, `last` — current market prices
- `volume` — day volume
- `impliedVolatility` — IB's calculated IV for this contract
- `modelGreeks` — a `Greeks` object with delta, gamma, theta, vega
- `callOpenInterest` / `putOpenInterest` — open interest

> Note: `modelGreeks` may be `None` if you don't have the right market data subscription or if the option hasn't traded.

In [ ]:
import time

rows = []
BATCH_SIZE = 50

for i in range(0, len(qualified), BATCH_SIZE):
    batch = qualified[i : i + BATCH_SIZE]
    tickers = ib.reqTickers(*batch)
    
    for ticker in tickers:
        c = ticker.contract
        g = ticker.modelGreeks   # None if no greeks available
        
        rows.append({
            'contract_symbol'   : c.localSymbol,
            'expiry'            : chosen_expiry,
            'strike'            : c.strike,
            'right'             : c.right,        # 'C' or 'P'
            'option_type'       : 'call' if c.right == 'C' else 'put',
            'bid'               : ticker.bid   if ticker.bid   and ticker.bid   > 0 else float('nan'),
            'ask'               : ticker.ask   if ticker.ask   and ticker.ask   > 0 else float('nan'),
            'last'              : ticker.last  if ticker.last  and ticker.last  > 0 else float('nan'),
            'volume'            : int(ticker.volume) if ticker.volume and ticker.volume >= 0 else None,
            'implied_volatility': ticker.impliedVolatility if ticker.impliedVolatility else float('nan'),
            # Greeks — all None if modelGreeks is not available
            'delta'             : g.delta if g else float('nan'),
            'gamma'             : g.gamma if g else float('nan'),
            'theta'             : g.theta if g else float('nan'),
            'vega'              : g.vega  if g else float('nan'),
            'open_interest'     : (
                ticker.callOpenInterest if c.right == 'C' else ticker.putOpenInterest
            ) or None,
        })
    
    print(f'Batch {i//BATCH_SIZE + 1}: fetched {len(batch)} contracts')
    time.sleep(0.5)   # respect IB pacing limits

print(f'\nTotal rows collected: {len(rows)}')

---
## Step 11 — Build a Clean DataFrame

Now we assemble a tidy DataFrame and clean up the types.

In [ ]:
df = pd.DataFrame(rows)

# Nullable integer for volume/OI (can be missing without becoming float)
df['volume']        = pd.array(df['volume'],        dtype='Int64')
df['open_interest'] = pd.array(df['open_interest'], dtype='Int64')

# Flag zero or missing IV — these rows exist but their IV is unreliable
df['iv_suspect'] = df['implied_volatility'].isna() | (df['implied_volatility'] == 0.0)

# Midpoint price
df['mid'] = (df['bid'] + df['ask']) / 2

print(f'Shape: {df.shape}')
print(f"Calls: {(df['option_type']=='call').sum()}  Puts: {(df['option_type']=='put').sum()}")
print(f"IV suspect rows: {df['iv_suspect'].sum()}")
print()
df.head(10)

---
## Step 12 — Disconnect

Always disconnect when you're done. TWS has a limit on concurrent connections (32 for most accounts).

In [ ]:
ib.disconnect()
print('Disconnected.')

---
## Step 13 — Explore the Data

No reconnection needed from here — we work with the DataFrame we already have.

In [ ]:
# Summary statistics
print('=== Calls ===')
print(df[df['option_type']=='call'][['strike','bid','ask','mid','implied_volatility','delta','open_interest']]
      .sort_values('strike').to_string(index=False))

In [ ]:
print('=== Puts ===')
print(df[df['option_type']=='put'][['strike','bid','ask','mid','implied_volatility','delta','open_interest']]
      .sort_values('strike').to_string(index=False))

---
## Step 14 — Visualise the IV Smile

The IV smile shows how implied volatility varies across strikes.  
- A flat smile = market assigns equal uncertainty at all strikes
- A downward skew (volatility skew) = OTM puts are more expensive than OTM calls → typical for equity indices
- Ag commodities often show an upward skew on calls (fear of supply shocks)

In [ ]:
calls = df[(df['option_type']=='call') & (~df['iv_suspect'])].sort_values('strike')
puts  = df[(df['option_type']=='put')  & (~df['iv_suspect'])].sort_values('strike')

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=calls['strike'], y=calls['implied_volatility'] * 100,
    mode='lines+markers', name='Calls IV',
    line=dict(color='#26a69a', width=2), marker=dict(size=6),
))
fig.add_trace(go.Scatter(
    x=puts['strike'], y=puts['implied_volatility'] * 100,
    mode='lines+markers', name='Puts IV',
    line=dict(color='#ef5350', width=2), marker=dict(size=6),
))
fig.add_vline(x=atm_price, line_dash='dash', line_color='white',
              annotation_text=f'ATM {atm_price:.2f}', annotation_position='top right')
fig.update_layout(
    title=f'IV Smile — {underlying.localSymbol} options expiry {chosen_expiry}',
    xaxis_title='Strike (USX/bu)',
    yaxis_title='Implied Volatility (%)',
    template='plotly_dark', height=420,
    legend=dict(orientation='h', y=1.05),
)
fig.show()

---
## Step 15 — Visualise the Greeks

In [ ]:
has_greeks = df['delta'].notna().any()

if not has_greeks:
    print('No greeks available — check your market data subscription in TWS.')
else:
    fig = make_subplots(rows=2, cols=2,
                        subplot_titles=['Delta', 'Gamma', 'Theta', 'Vega'])

    greek_plots = [
        ('delta', 1, 1), ('gamma', 1, 2), ('theta', 2, 1), ('vega', 2, 2)
    ]

    for greek, row, col in greek_plots:
        for side, name, color in [(calls, 'Call', '#26a69a'), (puts, 'Put', '#ef5350')]:
            fig.add_trace(go.Scatter(
                x=side['strike'], y=side[greek],
                mode='lines+markers', name=f'{name} {greek.capitalize()}',
                line=dict(color=color, width=2),
                showlegend=(row == 1 and col == 1),
            ), row=row, col=col)

    fig.update_layout(
        title=f'Option Greeks — {underlying.localSymbol} expiry {chosen_expiry}',
        template='plotly_dark', height=600,
        legend=dict(orientation='h', y=1.02),
    )
    fig.show()

---
## Step 16 — Build the Strike Ladder

A strike ladder shows calls and puts side by side, centred on ATM. This is the standard view in options trading platforms.

In [ ]:
call_cols = ['strike','bid','ask','mid','implied_volatility','delta','open_interest']
put_cols  = ['strike','bid','ask','mid','implied_volatility','delta','open_interest']

# Use only cols that exist and have data
call_cols = [c for c in call_cols if c in df.columns]
put_cols  = [c for c in put_cols  if c in df.columns]

call_side = (df[df['option_type']=='call'][call_cols]
             .sort_values('strike')
             .rename(columns={c: f'call_{c}' for c in call_cols if c != 'strike'})
             .set_index('strike'))

put_side  = (df[df['option_type']=='put'][put_cols]
             .sort_values('strike')
             .rename(columns={c: f'put_{c}'  for c in put_cols  if c != 'strike'})
             .set_index('strike'))

ladder = call_side.join(put_side, how='outer').reset_index()

# Format IV as percentage
for col in ['call_implied_volatility', 'put_implied_volatility']:
    if col in ladder.columns:
        ladder[col] = (ladder[col] * 100).round(2).astype(str) + '%'

# Highlight ATM row
ladder['atm'] = ladder['strike'].apply(lambda s: '← ATM' if abs(s - atm_price) == min(abs(ladder['strike'] - atm_price)) else '')

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 160)
print(ladder.to_string(index=False))

---
## Step 17 — Save to Parquet

Parquet preserves all dtypes cleanly (including nullable `Int64`) and is much faster to read/write than CSV for large option chains.

In [ ]:
from pathlib import Path
import sys

# Add project root to path so we can use ingestion.config
project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from ingestion.config import DATA_RAW_DIR

DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)
out_path = DATA_RAW_DIR / f'options_ZC_notebook_{chosen_expiry}.parquet'
df.to_parquet(out_path, index=False)

print(f'Saved {len(df)} rows to {out_path}')

# Verify round-trip
check = pd.read_parquet(out_path)
print(f'Read back: {len(check)} rows — OK')
print(check.dtypes)

---
## Summary — What Each IB API Call Does

| Call | Cost | Returns |
|---|---|---|
| `qualifyContracts(Future(...))` | Free | `conId`, `localSymbol`, expiry, multiplier |
| `reqTickers(future)` | 1 market data line | Spot price (bid/ask/last/volume) |
| `reqSecDefOptParams(...)` | Free | All valid (expiry, strike) pairs — no prices |
| `qualifyContracts(*options)` | Free | `conId` per option, drops invalid contracts |
| `reqTickers(*options)` | 1 line per option | Bid/ask/last/IV/greeks per contract |

## Key Gotchas

1. **`modelGreeks` can be `None`** — happens without the right data subscription, or for contracts with no recent trades
2. **`impliedVolatility == 0.0` ≠ missing** — deep ITM/OTM options sometimes return 0 instead of NaN
3. **Batch size ≤ 50** — IB will silently drop requests if you send too many at once
4. **Prices in cents** — ZC corn is quoted in US cents per bushel (USX/bu). Divide by 100 for USD.
5. **Front-month roll** — qualifying `Future('ZC', 'CBOT', 'USD')` without a specific date gives you the front month, which changes at roll time
6. **readonly=True** — enforces read-only at the protocol level; TWS rejects any order regardless of what code runs

## Next Steps in Crimson Vector

The data we pulled here feeds into:
- **Exotic pricing models** (Asian, barrier, lookback options) using the IV surface
- **Greeks surface** — plotting delta/vega across strikes and expiries
- **P&L attribution** — decomposing P&L into delta, gamma, theta, vega contributions